# Text classification with Crowdsourcing 

[Toloka](https://toloka.ai) is a crowdsourcing platform designed for constructing scalable pipelines of data labeling & data analysis. One of the main goals of Toloka is to support data-centric AI development.  
In order to get acquainted with **Toloka API & Interface** or **Toloka entities**, please visit our tutorial ["Crowdsourcing, Toloka, Introduction"](https://www.kaggle.com/edsukhodolskaya/crowdsourcing-toloka-introduction)

Let's start a series of tutorials covering the **NLP-related problems** and how they could be solved with the help of crowdsourcing. With Toloka, you can control data labeling accuracy to build a predictable pipeline of high-quality training data that impacts your NLP algorithms.

In this tutorial, we will explain how to create a **text classification** pipeline with Toloka.

Let's consider real-life example used for news moderation:  
*We have a set of news article headlines. We need to get these classified according to whether they are "clickbait"*.

>**Clickbait headline** is designed to make readers want to click on a hyperlink under the headline even if the link leads to content of dubious value (deceptive, sensationalized, or otherwise misleading).

## Prerequisites

Firstly prepare an environment and import all we'll need.

In [ ]:
!pip install toloka-kit==0.1.12
!pip install crowd-kit==0.0.5
!pip install pandas

import datetime
import sys
import time
import logging

import pandas
import numpy as np

import toloka.client as toloka
import toloka.client.project.template_builder as tb
from crowdkit.aggregation import DawidSkene

logging.basicConfig(
    format='[%(levelname)s] %(name)s: %(message)s',
    level=logging.INFO,
    stream=sys.stdout,
)

Then create a toloka-client instance. All API calls will go through it. You can learn about OAuth token & 'SANDBOX' environment in our tutorial ["Crowdsourcing, Toloka, Introduction"](https://www.kaggle.com/edsukhodolskaya/crowdsourcing-toloka-introduction)

In [ ]:
toloka_client = toloka.TolokaClient(input("Enter your token:"), 'PRODUCTION')  # Or switch to 'SANDBOX'
print(toloka_client.get_requester())

## Creating a project
Enter an attractive & clear project name and description.
> Note: A public project name and a public project description will be visible to the performers. We need to formulate our problem briefly and meaningfully to catch the crowd's attention.

In [ ]:
project = toloka.Project(
    public_name='Is this headline clickbait?',
    public_description='Look at the a news headline and decide if it is clickbait or not.',
)

Create the task interface for the project. 
> Check the [interface section](https://toloka.ai/knowledgebase/interface?utm_source=github&utm_medium=site&utm_campaign=tolokakit) of our Knowledge Base for more tips on interface design.

Our goal is to create a simple interface that will enable workers to see the headline and be able to classify it as 'clickbait' or 'not clickbait'

In [ ]:
text_viewer = tb.TextViewV1(tb.JoinHelperV1(['Headline: ', tb.InputData('headline')]))

radio_group_field = tb.ButtonRadioGroupFieldV1( #answers variety 
    tb.OutputData('category'),
    [
        tb.GroupFieldOption('clickbait', 'Clickbait'),
        tb.GroupFieldOption('notclickbait', 'Not clickbait'),
    ],
    validation=tb.RequiredConditionV1(hint='you need to select one answer'), #news' headlines can't be classified as both
)

task_width_plugin = tb.TolokaPluginV1(
    layout=tb.TolokaPluginV1.TolokaPluginLayout(
        kind='scroll',
        task_width=300,
    )
)

hot_keys_plugin = tb.HotkeysPluginV1( #introducing hotkeys is a good practice; they help to speed up the labeling process
    key_1=tb.SetActionV1(tb.OutputData('category'), 'clickbait'),
    key_2=tb.SetActionV1(tb.OutputData('category'), 'notclickbait'),
)

project_interface = toloka.project.TemplateBuilderViewSpec(
    view=tb.ListViewV1([text_viewer, radio_group_field]),
    plugins=[task_width_plugin, hot_keys_plugin],
)

For performers, our interface will look like this.

<table  align="center">
  <tr><td>
    <img src="https://github.com/Toloka/toloka-kit/blob/main/examples/5.nlp/text_classification/img/tasks_preview.png?raw=true"
         alt="Task page"  width="1000">
  </td></tr>
  <tr><td align="center">
    <b>Figure 1.</b> What the task page can looks like.
  </td></tr>
</table>

#### Input/output data specifications

Specifications describe a single unit of input data (f.e. type, data structure) used in a project and the single label in output data (f.e. type, data structure) collected from the performers.  

Here we are providing performers with a string (headline) and receiving a string label (either "clickbait" or "not clickbait")

In [ ]:
input_specification = {'headline': toloka.project.StringSpec()}
output_specification = {'category': toloka.project.StringSpec()}

project.task_spec = toloka.project.task_spec.TaskSpec(
    input_spec=input_specification,
    output_spec=output_specification,
    view_spec=project_interface,
)

#### Instructions

Write comprehensive instructions.  
Good instruction is the key to excellent performance!  
It should be easy, contain required definitions, and include examples and contr-examples if the problem is not intuitive.
> Get more tips on [designing instructions](https://toloka.ai/knowledgebase/instruction?utm_source=github&utm_medium=site&utm_campaign=tolokakit) in our Knowledge Base.


> It might be easier to use our user-friendly instruction editor in the Toloka interface for complex instructions.

In [ ]:
project.public_instructions = """<h2>About the task</h2>
In this task you need to classify headlines on 2 categories: Clickbait or Not clickbait.<br>
<h2>What is Clickbait headline?</h2>
Clickbait refers to the practice of writing sensationalized or misleading headlines.
<b>Clickbait headline<b> designed to make readers want to click on a hyperlink
especially when the link leads to content of dubious value. Typically clickbait titles cover not very useful content,
so visitors tend not to stay for too long, that's why it's bad.
"""

Create a project.

In [ ]:
project = toloka_client.create_project(project)

## Preparing data
This example uses SVM clickbait classifier, that distributed under a MIT license
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> Abhijnan Chakraborty, Bhargavi Paranjape, Sourya Kakarla, and Niloy Ganguly. "Stop Clickbait: Detecting and Preventing Clickbaits in Online News Media”. In Proceedings of the 2016 IEEE/ACM International Conference on Advances in Social Networks Analysis and Mining (ASONAM), San Fransisco, US, August 2016.


BibTex:
```
@inproceedings{chakraborty2016stop,
title={Stop Clickbait: Detecting and preventing clickbaits in online news media},
author={Chakraborty, Abhijnan and Paranjape, Bhargavi and Kakarla, Sourya and Ganguly, Niloy},
booktitle={Advances in Social Networks Analysis and Mining (ASONAM), 2016 IEEE/ACM International Conference on},
pages={9--16},
year={2016},
organization={IEEE}
}
```

Let's load this dataset and split it into three parts:
- training 
- golden
- main

1. **Training examples** will be used for training performers to classify topics correctly; we will provide them with corrected answers after reading the instruction and submitting the training. This is essential to set up a quality threshold for project completion.
2. **The golden dataset**, or so-called "honeypots", will be used as a validation dataset to check markup quality through the labeling process. We will mix production tasks which we seek to label with golden tasks, and evaluate performers' skill on this project based on their responses to honeypots. Performers can't distinguish between the golden and production tasks. 
3. **The main dataset** is the data that we want to label.

In [ ]:
!curl https://tlk.s3.yandex.net/ext_dataset/clickbait/clickbait_data.csv --output clickbait_data.csv
!curl https://tlk.s3.yandex.net/ext_dataset/clickbait/non_clickbait_data.csv --output non_clickbait_data.csv

clickbait_df = pandas.read_csv('clickbait_data.csv', sep='\t', names=['headline'])
clickbait_df['category'] = 'clickbait'
print(clickbait_df)

non_clickbait_df = pandas.read_csv('non_clickbait_data.csv', sep='\t', names=['headline'])
non_clickbait_df['category'] = 'notclickbait'
print(non_clickbait_df)

dataset = clickbait_df.append(non_clickbait_df)
dataset = dataset.sample(frac=1).reset_index(drop=True)

training_dataset, golden_dataset, main_dataset, _ = np.split(dataset, [10, 30, 130], axis=0)
print(f'\ntraining_dataset - {len(training_dataset)}')
print(f'\ngolden_dataset - {len(golden_dataset)}')
print(f'\nmain_dataset - {len(main_dataset)}')

## Creating a training pool
A pool is a set of tasks grouped into task pages.

Training is an essential part of almost every crowdsourcing project. It allows you to select performers who have mastered the task and thus improve quality. Training is also an excellent tool for scaling your task because you can run it when you need new performers on the project.

> Read more about [selecting performers](https://toloka.ai/knowledgebase/quality-control?utm_source=github&utm_medium=site&utm_campaign=tolokakit) in our Knowledge Base.

In [ ]:
training = toloka.Training(
    project_id=project.id,
    private_name='clickbait training',
    may_contain_adult_content=False,
    assignment_max_duration_seconds=60*30,
    mix_tasks_in_creation_order=False,
    shuffle_tasks_in_task_suite=False,
    training_tasks_in_task_suite_count=10, #number of training tasks per page
    task_suites_required_to_pass=10, #required number of training tasks to complete to receive a training skill
    retry_training_after_days=10,
    inherited_instructions=True,
)
training = toloka_client.create_training(training)

Upload training tasks to the pool.  
If the problem is not complex, as in our tutorial, there is no need to make the training pool large; it's enough to have 10-20 training tasks.

In [ ]:
training_tasks = [
    toloka.Task(
        pool_id=training.id,
        input_values={'headline': row['headline']},
        known_solutions = [toloka.task.BaseTask.KnownSolution(output_values={'category': row['category']})],
        message_on_unknown_solution=row['category'],
    )
    for _, row in training_dataset.iterrows()
]
result = toloka_client.create_tasks(training_tasks, allow_defaults=True)
print(len(result.items))

We recommend opening the training pool along with the main pool. Otherwise, Tolokers will spend their time on training but get no access to real tasks, which is frustrating. Also, do not forget to close the training pools when there are no main tasks available anymore.

## Creating a main pool
A main pool is a set of paid tasks grouped into task pages. Task pages are sent out for completion at the same time.

>Note: All tasks within a pool have the same settings (price, quality control, etc.)

Text classification tasks are generally paid as basic ones because these tasks do not take much time to complete. Read more about [pricing principles](https://toloka.ai/knowledgebase/pricing?utm_source=github&utm_medium=site&utm_campaign=tolokakit) in our Knowledge Base.

Sets an overlap of 3 (one atomic task - three labels from different performers) to get a more confident final label. To understand [how this rule works](https://toloka.ai/docs/guide/concepts/mvote.html?utm_source=github&utm_medium=site&utm_campaign=tolokakit), go to the Requester’s Guide.

Let's add a language filter so performers who speak English will be invited to complete this task. Then choose the Toloka web version and Toloka for mobile clients. These filters will allow performers to complete your task on their computers or mobile devices.

In [ ]:
pool = toloka.Pool(
    project_id=project.id,
    # Give the pool any convenient name. You are the only one who will see it.
    private_name='Is this headline clickbait?',
    may_contain_adult_content=False,
    # Set the price per task page.
    reward_per_assignment=0.01,
    will_expire=datetime.datetime.utcnow() + datetime.timedelta(days=365),
    # Overlap. This is the number of users who will complete the same task.
    defaults=toloka.Pool.Defaults(default_overlap_for_new_task_suites=3),
    # Time allowed for completing a task page
    assignment_max_duration_seconds=120,
    filter=(
        (toloka.filter.Languages.in_('EN')) &
        (
            (toloka.filter.ClientType == 'TOLOKA_APP') |
            (toloka.filter.ClientType == 'BROWSER')
        )
    ),
)

Attach the training you created earlier and select the accuracy level required to reach the main pool. This means Tolokers who got less than 90% accuracy or did not complete the training **will not have access to this pool**.

In [ ]:
pool.set_training_requirement(training_pool_id=training.id, training_passing_skill_value=90)

Set up [Quality control](https://toloka.ai/docs/guide/concepts/control.html?utm_source=github&utm_medium=site&utm_campaign=tolokakit):
 - Ban performers who give incorrect responses to golden tasks (since tasks like these have an answer that can be used as ground truth).
 - Set up the "Fast responses" rule. This rule allows you to ban performers who submit tasks at a suspiciously high speed.

Read more about [quality control principles](https://toloka.ai/knowledgebase/quality-control?utm_source=github&utm_medium=site&utm_campaign=tolokakit) in our Knowledge Base.

In [ ]:
pool.quality_control.add_action(
    collector=toloka.collectors.AssignmentSubmitTime(history_size=5, fast_submit_threshold_seconds=30),
    conditions=[toloka.conditions.FastSubmittedCount >= 2],
    action=toloka.actions.RestrictionV2(
        scope='POOL',
        duration_unit='PERMANENT',
        private_comment='bad quality'
    )
)

pool.quality_control.add_action(
    collector=toloka.collectors.GoldenSet(history_size=10), #compute answer rate on window of size 10
    conditions=[
        toloka.conditions.GoldenSetCorrectAnswersRate <= 90.0, #ban if this is true and
        toloka.conditions.GoldenSetAnswersCount >= 1 #performer completed at least one golden task
    ],
    action=toloka.actions.RestrictionV2(
        scope='POOL',
        duration_unit='PERMANENT',
        private_comment='bad quality'
    )
)

Specify	the number of tasks per page.  

We recommend putting as many tasks on one page as a performer can complete in 1 to 5 minutes. That way, performers are less likely to get tired, and they won’t lose a significant amount of data if a technical issue occurs.  
The recommended ratio of golden tasks per page is approximately 10 to 20 percent.

In [ ]:
pool.set_mixer_config(
    real_tasks_count=4,
    golden_tasks_count=1,
)

Create pool

In [ ]:
pool = toloka_client.create_pool(pool)

## Preparing and uploading tasks to the main pool

In [ ]:
golden_tasks = [
    toloka.task.Task(
        pool_id=pool.id,
        input_values={'headline': row['headline']},
        known_solutions = [
            toloka.task.BaseTask.KnownSolution(
                output_values={'category': row['category']}
            )
        ],
        infinite_overlap=True,
    )
    for _, row in golden_dataset.iterrows()
]

tasks = [
    toloka.task.Task(
        pool_id=pool.id,
        input_values={'headline': row['headline']},
    )
    for _, row in main_dataset.iterrows()
]

created_tasks = toloka_client.create_tasks(golden_tasks + tasks, allow_defaults=True)
print(len(created_tasks.items))

Start the main pool and the training pool.

**<font color='red'>Important</font>** Remember that real Toloka performers will complete the tasks.
**Double-check in Toloka interface** that everything is correct with your project configuration before you start the pool

In [ ]:
training = toloka_client.open_training(training.id)
print(training.status)
pool = toloka_client.open_pool(pool.id)
print(pool.status)

## Receiving responses

Wait until the pool is completed.

In [ ]:
pool_id = pool.id

def wait_pool_for_close(pool_id, minutes_to_wait=1):
    sleep_time = 60 * minutes_to_wait
    pool = toloka_client.get_pool(pool_id)
    while not pool.is_closed():
        op = toloka_client.get_analytics([toloka.analytics_request.CompletionPercentagePoolAnalytics(subject_id=pool.id)])
        op = toloka_client.wait_operation(op)
        percentage = op.details['value'][0]['result']['value']
        print(
            f'   {datetime.datetime.now().strftime("%H:%M:%S")}\t'
            f'Pool {pool.id} - {percentage}%'
        )
        time.sleep(sleep_time)
        pool = toloka_client.get_pool(pool.id)
    print('Pool was closed.')

wait_pool_for_close(pool_id)

When all the tasks are completed, get the raw responses from performers.

In [ ]:
answers = []

for assignment in toloka_client.get_assignments(pool_id=pool_id, status='ACCEPTED'):
    for task, solution in zip(assignment.tasks, assignment.solutions):
        if not task.known_solutions:
            answers.append([task.input_values['headline'], solution.output_values['category'], assignment.user_id])

print(f'answers count: {len(answers)}')

To get the final labeled dataset, we need to use the aggregation of received raw results.   
In this tutorial, we aggregate results using the **Dawid-Skene model**.

Read more about the [Dawid-Skene model](https://toloka.ai/docs/guide/concepts/result-aggregation.html?utm_source=github&utm_medium=site&utm_campaign=tolokakit#aggr__dawid-skene) in the Requester’s Guide or get at an overview of different [aggregation models](https://toloka.ai/knowledgebase/aggregation) our Knowledge Base.

In [ ]:
# Prepare dataframe
answers_df = pandas.DataFrame(answers, columns=['task', 'label', 'performer'])
# Run aggregation
predicted_answers = DawidSkene(n_iter=20).fit_predict(answers_df)

print(predicted_answers)

## Summary

This example showed how to solve classification problems with the help of crowdsourcing.

The described project is beneficial for:

- Creating benchmark datasets
- Monitoring the quality of your production classification model
- Evaluating the results of a more complex task, for example, machine translation 

Feel free to get to know Toloka & ask any questions!